# M2 · Segmentación de Instancias de Vehículos (`vehiculos.pt`)
Clases esperadas: `moto`, `auto`, `minibus`, `camion`, `bus`.
Configura en Google Colab el entorno de ejecución **GPU** y el secreto **`ROBOFLOW_API_KEY`**.

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow
import torch, ultralytics
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
from pathlib import Path

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
p = rf.workspace("TU_WORKSPACE").project("peaje-vehiculos-segmentacion")
dataset = p.version(1).download("yolov8")
yaml = Path(dataset.location) / "data.yaml"
print(yaml.read_text())

In [ ]:
from ultralytics import YOLO

m = YOLO("yolov8n-seg.pt")
m.train(
    data=str(yaml),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/peaje",
    name="m2",
    exist_ok=True,
    seed=42,
)

In [ ]:
from pathlib import Path
import json, shutil
from google.colab import drive
from ultralytics import YOLO

best = "/content/peaje/m2/weights/best.pt"
m = YOLO(best)
metrics = m.val(data=str(yaml), split="test")
print("mAP50 mascara:", metrics.seg.map50)
print("mAP50-95 mascara:", metrics.seg.map)

drive.mount("/content/drive")
out_dir = Path("/content/drive/MyDrive/smart-peaje-evidencias/m2")
out_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(best, "/content/drive/MyDrive/vehiculos.pt")
shutil.copy(best, out_dir / "vehiculos.pt")
for fname in ["results.csv", "results.png", "confusion_matrix.png", "MaskPR_curve.png", "MaskF1_curve.png"]:
    src = Path("/content/peaje/m2") / fname
    if src.is_file():
        shutil.copy(src, out_dir / fname)

(out_dir / "metricas_m2.json").write_text(
    json.dumps({
        "map50_mask": float(metrics.seg.map50),
        "map50_95_mask": float(metrics.seg.map),
        "map50_box": float(metrics.box.map50),
        "map50_95_box": float(metrics.box.map),
    }, indent=2),
    encoding="utf-8",
)
print("Peso y evidencias guardados en Google Drive.")